# 3D U-Net + FADC-Encoder V2.2 + Attention Diversity Aux Loss (four-attentions wired)

**v2.2 iteration on the attention-diversity branch.** Rebuilds on the stabilised v2.1 run but with the four-attentions wiring fix:

- `s_att` **enabled** — OmniAttention3DSpatial is now constructed with the real conv kernel_size, so `spatial_fc` exists and s_att is a live `(B,1,1,1,1,K,K,K)` tensor (was hard-coded to kernel_size=1 → SKIP).
- `s_att` **applied** — grouped `F.conv3d` per branch multiplies `conv.weight * (s_att * 2)` per sample, preserving stride / padding / dilation / groups / bias.
- `f_att` **moved post-BN** — pre-BN scaling was being normalised away (diagnostic: max input std ~0.0022). `FADCConvBlockV2` now runs `bn(conv(x)) * (f_att * 2)`.
- Aux loss **extended** to s_att (batch-std hinge) and k_att (spatial-mean per branch, then batch-std hinge — targets input-adaptivity of branch preference, not per-voxel texture).

Because s_att and post-BN f_att are now live at EVERY FADC layer, the `--attn_diversity_layers` filter is set to `''` (all layers), not the v2.1 `enc3,enc4` deep-only filter.

**Changes vs v2.1 stabilised iteration:**

| Setting | v2.1 stabilised | v2.2 | Reason |
|---|---|---|---|
| `spatial_fc` in OmniAttention3DSpatial | None (SKIP) | live tensor | Enables 4th attention head |
| f_att application | pre-BN inside conv | POST-BN in FADCConvBlockV2 | BN doesn't wash it out |
| aux loss inputs | c_att, f_att | c/f/s/k | All four heads receive diversity pressure |
| `--attn_diversity_layers` | 'enc3,enc4' | **''** (all) | s_att is new everywhere, needs the push |
| `--attn_diversity_weight` | 0.005 | 0.005 | unchanged (soft push) |
| `--k_att_temp_end` | 0.8 | 0.8 | unchanged |
| `--attn_diversity_start_epoch` | 10 | 10 | unchanged |

**Reference numbers to beat:**

| Reference | Val Dice | Notes |
|---|---|---|
| v2.1 stabilised s=42 ep30 | 0.5255 | previous attempt — aux worked briefly then collapsed |
| v2 fixed Encoder s=42 ep70 | 0.6267 | v2 plateau |
| Baseline UNet3D 2ch | 0.6735 | uncontrolled seed |
| v1 Bottleneck s=42 | 0.6801 | project ceiling |

**Diagnostic reads to watch (post-train cell 8):**
1. `s inp-std` should be > 0.005 at multiple layers (proves s_att is not idle — the new v2.2 head).
2. `f inp-std` at deeper layers should be > 0.005 (proves the post-BN move fixed the wash-out).
3. `k inp-std` (spatial-mean of branch-0 across inputs) should be > 0.005 (proves aux loss reached k_att).
4. `aux_loss` in the per-epoch summary should be 0 for the first 10 epochs, then non-zero and DECREASING.

**Setup:** FADC at encoder placement, 100 epochs, batch 2, seed 42, cudnn deterministic, val_every=10.

**GPU:** Kaggle T4 x 2. ~7 h expected wall time.

**IMPORTANT:** Download `best_model.pth`, `train_log.json`, `meta.json` at every val_every landing — `/kaggle/working` wipes on session close.

In [ ]:
# CONFIG (v2.2 — four-attentions wired)
SEED = 42

DATA_ROOT              = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
OUTPUT_DIR             = f"/kaggle/working/outputs/fadc_encoder_diversity_v22_2ch_100ep_s{SEED}"
CODE_DIR               = "/kaggle/working/FADC-3D"
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

EPOCHS       = 100
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [96, 96, 48]
WARMUP       = 5
VAL_EVERY    = 10

# k_att schedule — unchanged from v2.1 stabilised iteration.
K_ATT_TEMP_START    = 4.0
K_ATT_TEMP_END      = 0.8
K_ATT_ANNEAL_EPOCHS = 60

# Attention diversity aux loss — v2.2: applied to ALL layers now that s_att
# and post-BN f_att are live everywhere. Weight/target/start_epoch unchanged.
ATTN_DIVERSITY_WEIGHT       = 0.005
ATTN_DIVERSITY_TARGET       = 0.03
ATTN_DIVERSITY_START_EPOCH  = 10
ATTN_DIVERSITY_LAYERS       = ''           # v2.2: empty string = all FADC layers

RESUME_FROM  = ""
GIT_BRANCH   = "feature/attention-diversity-loss"

print(f"SEED                    : {SEED}")
print(f"OUTPUT_DIR              : {OUTPUT_DIR}")
print(f"BRANCH                  : {GIT_BRANCH}")
print(f"k_att temp              : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"attn_diversity_weight   : {ATTN_DIVERSITY_WEIGHT}")
print(f"attn_diversity_target   : {ATTN_DIVERSITY_TARGET}")
print(f"attn_diversity_start_ep : {ATTN_DIVERSITY_START_EPOCH}")
print(f"attn_diversity_layers   : '{ATTN_DIVERSITY_LAYERS}' (empty = all FADC layers, v2.2)")
print(f"val_every               : {VAL_EVERY}")

In [ ]:
# 1. INSTALL DEPENDENCIES
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    p = torch.cuda.get_device_properties(0)
    print(f"VRAM           : {p.total_memory / 1e9:.1f} GB")

In [ ]:
# 2. CLONE / UPDATE ATTENTION-DIVERSITY BRANCH
import os, sys
if os.path.exists(CODE_DIR):
    print(f"Repo exists — fetching + checkout {GIT_BRANCH} ...")
    os.system(f"git -C {CODE_DIR} fetch --all")
    os.system(f"git -C {CODE_DIR} checkout {GIT_BRANCH}")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone -b {GIT_BRANCH} https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
sys.path.insert(0, CODE_DIR)

for p in ["fadc_3d_v2/omni_attention_3d_spatial.py",
         "models/unet_3d_fadc_v2.py",
         "training/train_centralized_v2.py"]:
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing: {p}"
print("Modules present.")

In [ ]:
# 3. PULL VERIFIER — abort if v2.2 (four-attentions) code did not land
# Guards against wasting 7h if git pull failed or the branch tip is stale.
import inspect, sys, subprocess
sys.path.insert(0, CODE_DIR)
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2
from models.unet_3d_fadc_v2 import FADCConvBlockV2
from training import train_centralized_v2 as train_mod

# Print current branch tip
r = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"],
                   capture_output=True, text=True)
print(f"HEAD  : {r.stdout.strip()}")

# --- V2 baseline checks (must still hold on this branch) ---
ch_src = inspect.getsource(OmniAttention3DSpatial.get_channel_attention)
fi_src = inspect.getsource(OmniAttention3DSpatial.get_filter_attention)
ka_src = inspect.getsource(OmniAttention3DSpatial.get_kernel_attention_spatial)
assert 'self.temperature' not in ch_src, 'v2 temperature-scope fix missing (get_channel_attention)'
assert 'self.temperature' not in fi_src, 'v2 temperature-scope fix missing (get_filter_attention)'
assert 'self.temperature' in ka_src,     'k_att lost its temperature term'
print('  [OK] v2 temperature-scope fix intact')

# --- V2.1 avg+max concat + always-on filter_fc ---
init_src = inspect.getsource(OmniAttention3DSpatial.__init__)
fwd_src  = inspect.getsource(OmniAttention3DSpatial.forward)
assert 'AdaptiveMaxPool3d' in init_src, 'v2.1 max pool NOT in __init__'
assert 'torch.cat' in fwd_src,          'v2.1 avg+max concat NOT in forward'
assert 'if in_planes == groups' not in init_src, 'filter_fc skip still present'
print('  [OK] v2.1 avg + max concat + always-on filter_fc')

# --- V2.2 four-attentions gates: THE CRITICAL ADDS ---
# (a) s_att is LIVE: constructing AdaptiveDilatedConv3DV2 with kernel_size=3
#     should produce a spatial_fc conv (not None) on the inner OmniAttention.
_probe = AdaptiveDilatedConv3DV2(4, 8, kernel_size=3)
assert _probe.omni_att.spatial_fc is not None, \
    'v2.2 s_att gate FAILED: spatial_fc is None. AdaptiveDilatedConv3DV2 is still ' \
    'passing kernel_size=1 to OmniAttention3DSpatial. Branch has NOT been updated to v2.2.'
print('  [OK] v2.2 s_att enabled — spatial_fc exists on omni_att')

# (b) f_att is applied POST-BN: FADCConvBlockV2 must build inner convs with
#     apply_filter_attention=False and expose last_filter_attention.
_blk = FADCConvBlockV2(4, 8)
assert _blk.conv1.apply_filter_attention is False, \
    'v2.2 f_att gate FAILED: FADCConvBlockV2.conv1.apply_filter_attention is True. ' \
    'Branch has NOT been updated to move f_att post-BN.'
assert hasattr(_blk.conv1, 'last_filter_attention'), \
    'v2.2 f_att gate FAILED: last_filter_attention not exposed on AdaptiveDilatedConv3DV2.'
print('  [OK] v2.2 f_att moved post-BN — FADCConvBlockV2 conv1.apply_filter_attention=False')

# (c) Aux loss extended to s_att and k_att.
store_src = inspect.getsource(train_mod._AttnStore)
loss_src  = inspect.getsource(train_mod.attention_diversity_loss)
assert 's_atts' in store_src and 'k_atts' in store_src, \
    'v2.2 aux gate FAILED: _AttnStore does not collect s_atts/k_atts. Branch stale.'
assert 's_atts' in loss_src and 'k_atts' in loss_src, \
    'v2.2 aux gate FAILED: attention_diversity_loss does not use s_atts/k_atts. Branch stale.'
print('  [OK] v2.2 aux loss extended to s_att and k_att')

# --- Attention diversity aux loss helpers ---
assert hasattr(train_mod, 'register_attention_hooks'), 'register_attention_hooks missing'
assert hasattr(train_mod, 'attention_diversity_loss'), 'attention_diversity_loss missing'
assert hasattr(train_mod, '_AttnStore'), '_AttnStore missing'
assert hasattr(train_mod, '_name_matches_prefixes'), '_name_matches_prefixes missing'
print('  [OK] attention diversity helpers present (incl. layer-prefix matcher)')

# --- register_attention_hooks now accepts layer_prefixes ---
sig = inspect.signature(train_mod.register_attention_hooks)
assert 'layer_prefixes' in sig.parameters, \
    'register_attention_hooks missing layer_prefixes kwarg'
print('  [OK] register_attention_hooks(model, layer_prefixes=...) present')

# --- CLI args landed ---
help_out = subprocess.run(
    [sys.executable, os.path.join(CODE_DIR, 'training', 'train_centralized_v2.py'), '--help'],
    capture_output=True, text=True).stdout
for arg in ('--attn_diversity_weight', '--attn_diversity_target',
            '--attn_diversity_start_epoch', '--attn_diversity_layers'):
    assert arg in help_out, f'{arg} CLI missing'
print('  [OK] all four attn_diversity CLI args are wired')

print()
print('Verification passed — v2.2 four-attentions branch pulled correctly.')

In [ ]:
# 4. ARCH SMOKE — v2.2 encoder + all-layer aux loss end-to-end
# Tests that (a) the empty-filter picks ALL 8 encoder FADC layers (v2.2),
# (b) aux_loss has grad_fn and collects c/f/s/k, (c) backward reaches
# channel_fc / filter_fc / spatial_fc / kernel_spatial_head at enc4.c2.
import torch
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial
from training.train_centralized_v2 import register_attention_hooks, attention_diversity_loss

device = torch.device('cuda')
model = UNet3DFADC_V2(in_channels=2, out_channels=2, base_filters=32,
                       fadc_placement='encoder').to(device).train()
n_params = sum(p.numel() for p in model.parameters())
print(f"Params           : {n_params:,}")

# v2.1 shape signature check
for name, m in model.named_modules():
    if isinstance(m, OmniAttention3DSpatial):
        in_planes = m.channel_fc.out_channels
        assert m.fc.in_channels == 2 * in_planes, f"v2.1 avg+max concat missing at {name}"
        assert m.filter_fc is not None, f"filter_fc missing at {name}"
        assert m.spatial_fc is not None, f"v2.2 spatial_fc missing at {name}"
print('  [OK] v2.1 fc concat + v2.2 spatial_fc live at every OmniAttention layer')

# v2.2 gate: FADCConvBlockV2 must apply f_att post-BN on every inner conv.
for name, m in model.named_modules():
    if isinstance(m, AdaptiveDilatedConv3DV2):
        assert m.apply_filter_attention is False, \
            f"apply_filter_attention still True at {name} — post-BN move did not land"
print('  [OK] every FADC inner conv has apply_filter_attention=False (v2.2 post-BN wiring)')

# Register aux hooks with the layer filter that matches this run's config.
# v2.2: ATTN_DIVERSITY_LAYERS='' -> None -> hook every FADC layer (8 for encoder).
layer_prefixes = [p.strip() for p in ATTN_DIVERSITY_LAYERS.split(',') if p.strip()]
store, handles, hooked_names = register_attention_hooks(model,
                                                        layer_prefixes=layer_prefixes or None)
print(f'  [OK] hooks registered on {len(handles)} layer(s)')
if not layer_prefixes:
    assert len(handles) == 8, f"expected 8 hooks (all 4 encoder blocks x conv1/conv2), got {len(handles)}"
    assert all('enc' in n for n in hooked_names), f"unexpected non-encoder hooks: {hooked_names}"
else:
    # Backwards-compatible mode: matches old enc3,enc4 filter.
    assert len(handles) == 4 * len(layer_prefixes) // 2 * 2, \
        f"expected 2 hooks per prefix, got {len(handles)}"

# Forward with B=2 so batch-std is meaningful
x = torch.randn(2, 2, *PATCH_SIZE, device=device)
target = torch.zeros(2, *PATCH_SIZE, device=device, dtype=torch.long)

store.clear()
store.enabled = True
y = model(x)
store.enabled = False
print(f"forward OK. output shape: {tuple(y.shape)}")
print(f"store captured: c={len(store.c_atts)} f={len(store.f_atts)} "
      f"s={len(store.s_atts)} k={len(store.k_atts)}   (expect {len(handles)} each for v2.2)")
assert len(store.c_atts) == len(handles) and len(store.f_atts) == len(handles), \
    'c/f capture mismatch — hook or store bug'
assert len(store.s_atts) == len(handles), \
    'v2.2 gate FAILED: s_att not captured. Is spatial_fc still None on some layer?'
assert len(store.k_atts) == len(handles), \
    'v2.2 gate FAILED: k_att not captured.'

aux = attention_diversity_loss(store, target_std=ATTN_DIVERSITY_TARGET, device=device)
print(f"aux_loss = {aux.item():.6f}   grad_fn = {aux.grad_fn}")
assert aux.grad_fn is not None, 'aux_loss lost its grad_fn — backprop will not flow'

# Confirm backward reaches all four heads at the deepest layer
seg_loss = torch.nn.functional.cross_entropy(y, target)
total = seg_loss + ATTN_DIVERSITY_WEIGHT * aux

deep_layer = None
for name, mod in model.named_modules():
    if name.endswith('enc4.conv.conv2.omni_att'):
        deep_layer = mod; break
assert deep_layer is not None

model.zero_grad()
total.backward()
assert deep_layer.channel_fc.weight.grad         is not None, 'no grad on channel_fc.weight at enc4.c2'
assert deep_layer.filter_fc.weight.grad          is not None, 'no grad on filter_fc.weight at enc4.c2'
assert deep_layer.spatial_fc.weight.grad         is not None, 'v2.2 gate: no grad on spatial_fc.weight at enc4.c2'
assert deep_layer.kernel_spatial_head.weight.grad is not None, 'no grad on kernel_spatial_head.weight at enc4.c2'
print('  [OK] backward reached c/f/s/k heads at enc4.conv.conv2 (all four attentions live)')

for h in handles: h.remove()
del model, x, y, aux, seg_loss, total
torch.cuda.empty_cache()
print('\nv2.2 encoder + all-layer aux smoke test PASSED.')

In [ ]:
# 5. CACHE SANITY
import os, numpy as np
from pathlib import Path

cache_path = Path(PREPROCESSED_CACHE_DIR)
assert cache_path.exists(), f"Cache not found: {cache_path}"
train_npzs = sorted((cache_path / "train").glob("*.npz"))
val_npzs   = sorted((cache_path / "val").glob("*.npz"))
print(f"Train : {len(train_npzs)} | Val: {len(val_npzs)}")
for p in [train_npzs[0], train_npzs[-1], val_npzs[0]]:
    d = np.load(p)
    print(f"  {p.name}  image={d['image'].shape}  label={d['label'].shape}")
    assert d['image'].shape[0] == 2, f"NOT 2-channel: {p.name}"
print("Cache OK.")

In [ ]:
# 6. LAUNCH TRAINING
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized_v2.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",          "unet3d_fadc_encoder_v2",
    "--data_root",      DATA_ROOT,
    "--output_dir",     OUTPUT_DIR,
    "--epochs",         str(EPOCHS),
    "--batch_size",     str(BATCH_SIZE),
    "--num_workers",    str(NUM_WORKERS),
    "--patch_size",     str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",  str(WARMUP),
    "--val_every",      str(VAL_EVERY),
    "--seed",           str(SEED),
    "--k_att_temp_start",           str(K_ATT_TEMP_START),
    "--k_att_temp_end",             str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",        str(K_ATT_ANNEAL_EPOCHS),
    "--attn_diversity_weight",      str(ATTN_DIVERSITY_WEIGHT),
    "--attn_diversity_target",      str(ATTN_DIVERSITY_TARGET),
    "--attn_diversity_start_epoch", str(ATTN_DIVERSITY_START_EPOCH),
    "--attn_diversity_layers",      ATTN_DIVERSITY_LAYERS,
]
if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]
if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# 7. TRAINING CURVES — Loss, Val Dice, Aux loss, k_att temperature
import json, os
import matplotlib.pyplot as plt

BASELINE_DICE       = 0.6735
V2_FIXED_EP70       = 0.6267
V21_EP30            = 0.5247    # Encoder v2.1 s=42 ep30 (this branch's direct comparator)
BOTTLENECK_V1_DICE  = 0.6801

log_path = os.path.join(OUTPUT_DIR, "train_log.json")
if not os.path.exists(log_path):
    print("No training log yet.")
else:
    with open(log_path) as f:
        log = json.load(f)
    epochs     = [e["epoch"] for e in log]
    losses     = [e["loss"]  for e in log]
    aux_losses = [e.get("aux_loss", 0.0) for e in log]
    temps      = [e.get("k_att_temperature", float("nan")) for e in log]
    val_epochs = [e["epoch"]    for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    ax.plot(epochs, losses, color="steelblue")
    ax.set_title("Total Training Loss"); ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(val_epochs, val_dices, color="darkorange", marker="o", markersize=4, label="this run")
    ax.axhline(V21_EP30,          color="gold",   ls="-",  label=f"v2.1 s=42 ep30 ({V21_EP30:.4f})")
    ax.axhline(V2_FIXED_EP70,     color="brown",  ls="-",  label=f"v2 fixed ep70 ({V2_FIXED_EP70:.4f})")
    ax.axhline(BASELINE_DICE,     color="green",  ls="--", label=f"Baseline ({BASELINE_DICE:.4f})")
    ax.axhline(BOTTLENECK_V1_DICE, color="purple", ls=":",  label=f"v1 Bottleneck s=42 ({BOTTLENECK_V1_DICE:.4f})")
    if val_dices:
        ax.set_title(f"Val Dice  (best: {max(val_dices):.4f})")
    else:
        ax.set_title("Val Dice")
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(epochs, aux_losses, color="crimson")
    ax.axhline(0.0, color="gray", ls=":")
    ax.set_title("Attention Diversity Aux Loss (hinge; should DROP as std grows)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("aux_loss"); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(epochs, temps, color="darkred")
    ax.set_title(f"k_att T ({K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("T"); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()

    if val_dices:
        best = max(val_dices)
        print(f"Best Val Dice            : {best:.4f}")
        print(f"vs v2.1 s=42 ep30        : {best - V21_EP30:+.4f}   (direct comparator)")
        print(f"vs v2 fixed ep70         : {best - V2_FIXED_EP70:+.4f}")
        print(f"vs Baseline              : {best - BASELINE_DICE:+.4f}")
        print(f"vs v1 Bottleneck s=42    : {best - BOTTLENECK_V1_DICE:+.4f}")

    if aux_losses:
        print(f"\nAux loss: start {aux_losses[0]:.4f}  -> end {aux_losses[-1]:.4f}")
        print(f"(hinge target {ATTN_DIVERSITY_TARGET}; aux dropping = c_att/f_att batch-std growing)")

In [ ]:
# 8. POST-TRAIN DIAGNOSTIC — c / f / s / k adaptation on the best_model.pth
# v2.2 extension: also reports s_att (new head) and k_att per-input adaptivity.
import os, sys, torch, math
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial

ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
if not os.path.exists(ckpt_path):
    print("No best_model.pth — run training first.")
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt['config']
    epoch = ckpt.get('epoch', 0)
    print(f"Loaded ckpt: epoch={epoch} (0-idx), best_dice={ckpt.get('best_dice'):.4f}")

    model = UNet3DFADC_V2(in_channels=cfg['model']['in_channels'],
                          out_channels=cfg['model']['out_channels'],
                          base_filters=cfg['model']['base_filters'],
                          fadc_placement='encoder').to(device).eval()
    model.load_state_dict(ckpt['model'])

    # Set the temperature the training loop would have set at that epoch
    def est_T(e, ae=K_ATT_ANNEAL_EPOCHS, ts=K_ATT_TEMP_START, te=K_ATT_TEMP_END):
        if ae <= 1: return te
        e = min(max(e, 0), ae - 1)
        c = 0.5 * (1.0 + math.cos(math.pi * e / (ae - 1)))
        return te + (ts - te) * c
    T_est = est_T(epoch)
    if hasattr(model, 'set_temperature'):
        model.set_temperature(T_est)
    print(f"Estimated k_att T at this epoch : {T_est:.3f}")

    # Capture c / f / s / k from every OmniAttention3DSpatial. s_att may be
    # a scalar 1.0 on legacy ckpts trained before v2.2 (spatial_fc=None) —
    # we tolerate that and mark those layers SKIP.
    capture = {}
    handles = []
    for name, mod in model.named_modules():
        if isinstance(mod, OmniAttention3DSpatial):
            capture[name] = {'c': [], 'f': [], 's': [], 'k': []}
            def mk(nm):
                def hook(_m, _i, out):
                    c_att, f_att, s_att, k_att = out
                    capture[nm]['c'].append(c_att.detach().cpu())
                    capture[nm]['f'].append(f_att.detach().cpu())
                    if torch.is_tensor(s_att):
                        capture[nm]['s'].append(s_att.detach().cpu())
                    if torch.is_tensor(k_att):
                        capture[nm]['k'].append(k_att.detach().cpu())
                return hook
            handles.append(mod.register_forward_hook(mk(name)))

    # Feed 8 different random inputs (each batch=1 -> N=8 samples per layer)
    torch.manual_seed(0)
    N = 8
    with torch.no_grad():
        for _ in range(N):
            x = torch.randn(1, 2, 32, 32, 16, device=device)
            _ = model(x)
    for h in handles: h.remove()

    header = (f"{'Layer':<28} "
              f"{'c mean':>8} {'c inp-std':>10} "
              f"{'f mean':>8} {'f inp-std':>10} "
              f"{'s inp-std':>10} "
              f"{'k spat-std':>11} {'k inp-std':>10}")
    print()
    print('=' * len(header))
    print(header)
    print('-' * len(header))
    for name in sorted(capture):
        c = torch.stack(capture[name]['c'], 0)
        f = torch.stack(capture[name]['f'], 0)
        c_mean = c.mean().item()
        f_mean = f.mean().item()
        c_inp  = c.flatten(2).mean(dim=2).std().item()
        f_inp  = f.flatten(2).mean(dim=2).std().item()

        if capture[name]['s']:
            s = torch.stack(capture[name]['s'], 0)
            s_inp_s = f"{s.flatten(2).mean(dim=2).std().item():.4f}"
        else:
            s_inp_s = "SKIP"

        if capture[name]['k']:
            k = torch.stack(capture[name]['k'], 0)   # (N, 1, n_br, D, H, W)
            spat = k.float().flatten(3).std(dim=3, unbiased=False).mean().item()
            per_input = k[:, 0, 0].flatten(1).mean(dim=1).std().item()
            k_spat_s = f"{spat:.4f}"
            k_inp_s  = f"{per_input:.4f}"
        else:
            k_spat_s = "SKIP"
            k_inp_s  = "SKIP"

        print(f"{name:<28} "
              f"{c_mean:>8.4f} {c_inp:>10.4f} "
              f"{f_mean:>8.4f} {f_inp:>10.4f} "
              f"{s_inp_s:>10} "
              f"{k_spat_s:>11} {k_inp_s:>10}")
    print('=' * len(header))

    print()
    print('Reading guide (v2.2):')
    print('  c/f inp-std > 0.005  -> attention IS varying across inputs (working)')
    print('  s inp-std   > 0.005  -> new v2.2 s_att head is input-adaptive')
    print('  k inp-std   > 0.005  -> aux loss reached k_att branch preference')
    print()
    print('Baseline for comparison (v2.1 s=42 ep30, WITHOUT s_att & post-BN f_att):')
    print('  enc4.c2 c_att std : 0.0002')
    print('  enc3.c1 c_att std : 0.0022')
    print('  enc4.c2 k_att per-input-std : 0.0174')

In [ ]:
# 9. DOWNLOAD LINKS
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "latest_checkpoint.pth", "train_log.json", "meta.json", "training_curves.png"):
    p = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")